# Rainfall Prediction Classifier

Predicts whether it will rain **today** in the Melbourne area using next-day-available historical weather observations (temperature, humidity, pressure, wind, cloud cover, season, etc.) from the Australian Bureau of Meteorology dataset (2008–2017).

Two models are trained and compared inside a single scikit-learn pipeline: a **Random Forest Classifier** and a **Logistic Regression** classifier, both tuned with `GridSearchCV` and stratified cross-validation.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay


## Load and clean the data

In [ ]:
url = "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/_0eYOqji3unP1tDNKWZMjg/weatherAUS-2.csv"
df = pd.read_csv(url)

# Drop rows with missing values (still leaves ~56k observations)
df = df.dropna()

# Reframe the problem: predict TODAY's rain using data available up to and including yesterday,
# which avoids using same-day information that wouldn't be available at prediction time.
df = df.rename(columns={'RainToday': 'RainYesterday', 'RainTomorrow': 'RainToday'})

# Focus on three nearby locations (Melbourne, Melbourne Airport, Watsonia) to keep weather patterns consistent
df = df[df.Location.isin(['Melbourne', 'MelbourneAirport', 'Watsonia'])]

df.shape


## Feature engineering: seasonality

In [ ]:
def date_to_season(date):
    month = date.month
    if month in (12, 1, 2):
        return 'Summer'
    elif month in (3, 4, 5):
        return 'Autumn'
    elif month in (6, 7, 8):
        return 'Winter'
    else:
        return 'Spring'

df['Date'] = pd.to_datetime(df['Date'])
df['Season'] = df['Date'].apply(date_to_season)
df = df.drop(columns=['Date'])


## Train/test split

In [ ]:
X = df.drop(columns='RainToday')
y = df['RainToday']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


## Preprocessing pipeline

In [ ]:
numeric_features = X_train.select_dtypes(include=['number']).columns.tolist()
categorical_features = X_train.select_dtypes(include=['object', 'category']).columns.tolist()

numeric_transformer = Pipeline(steps=[('scaler', StandardScaler())])
categorical_transformer = Pipeline(steps=[('onehot', OneHotEncoder(handle_unknown='ignore'))])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])


## Model 1: Random Forest (tuned with GridSearchCV)

In [ ]:
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

param_grid_rf = {
    'classifier__n_estimators': [50, 100],
    'classifier__max_depth': [None, 10, 20],
    'classifier__min_samples_split': [2, 5]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search_rf = GridSearchCV(pipeline, param_grid_rf, cv=cv, scoring='accuracy', verbose=0)
grid_search_rf.fit(X_train, y_train)

print("Best parameters:", grid_search_rf.best_params_)
print("Best CV accuracy: {:.3f}".format(grid_search_rf.best_score_))


### Random Forest — results

In [ ]:
y_pred_rf = grid_search_rf.predict(X_test)
test_score_rf = grid_search_rf.score(X_test, y_test)

print("Test set accuracy: {:.3f}\n".format(test_score_rf))
print(classification_report(y_test, y_pred_rf))

conf_matrix_rf = confusion_matrix(y_test, y_pred_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix_rf, display_labels=grid_search_rf.classes_)
disp.plot(cmap='Blues')
plt.title('Random Forest — Confusion Matrix')
plt.show()


### Feature importance

In [ ]:
feature_names = numeric_features + list(
    grid_search_rf.best_estimator_['preprocessor']
    .named_transformers_['cat']
    .named_steps['onehot']
    .get_feature_names_out(categorical_features)
)

feature_importances = grid_search_rf.best_estimator_['classifier'].feature_importances_
importance_df = pd.DataFrame({
    'Feature': feature_names,
    'Importance': feature_importances
}).sort_values(by='Importance', ascending=False)

N = 15
top_features = importance_df.head(N)

plt.figure(figsize=(9, 6))
plt.barh(top_features['Feature'], top_features['Importance'], color='skyblue')
plt.gca().invert_yaxis()
plt.title(f'Top {N} Features — Predicting Rain Today')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()


## Model 2: Logistic Regression (tuned with GridSearchCV)

In [ ]:
pipeline.set_params(classifier=LogisticRegression(random_state=42, max_iter=1000))

param_grid_lr = {
    'classifier__solver': ['liblinear'],
    'classifier__penalty': ['l1', 'l2'],
    'classifier__class_weight': [None, 'balanced']
}

grid_search_lr = GridSearchCV(pipeline, param_grid_lr, cv=cv, scoring='accuracy', verbose=0)
grid_search_lr.fit(X_train, y_train)

print("Best parameters:", grid_search_lr.best_params_)
print("Best CV accuracy: {:.3f}".format(grid_search_lr.best_score_))


### Logistic Regression — results

In [ ]:
y_pred_lr = grid_search_lr.predict(X_test)
test_score_lr = grid_search_lr.score(X_test, y_test)

print("Test set accuracy: {:.3f}\n".format(test_score_lr))
print(classification_report(y_test, y_pred_lr))

conf_matrix_lr = confusion_matrix(y_test, y_pred_lr)
disp = ConfusionMatrixDisplay(confusion_matrix=conf_matrix_lr, display_labels=grid_search_lr.classes_)
disp.plot(cmap='Blues')
plt.title('Logistic Regression — Confusion Matrix')
plt.show()


## Model comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Random Forest', 'Logistic Regression'],
    'Test Accuracy': [test_score_rf, test_score_lr],
    'CV Accuracy': [grid_search_rf.best_score_, grid_search_lr.best_score_]
})
comparison
